In [ ]:
# INSTALL REQUIRED LIBRARIES
import os
!pip install azure-ai-textanalytics 
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
import matplotlib.pyplot as plt
import pandas as pd

# Define Azure Credentials
API_KEY = "EftX6P6wcO5eU5lqduZnG9dpibmXe1lQFqaOs8boVOVfIAOSbP5gJQQJ99CBACYeBjFXJ3w3AAAaACOGfjen" # Repace here Own Key
ENDPOINT = "https://lang-sentiment-lab.cognitiveservices.azure.com/"  # Replace here Own Endpoint

# Initialize client
client = TextAnalyticsClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(API_KEY)
)

# Input text data (10 real-world examples including mixed sentiment)
documents = [
    "Love this product! Works perfectly and fast delivery.",
    "Terrible experience. Broke after 2 days. Avoid this brand.",
    "Average quality. Does the job but nothing special.",
    "Absolutely fantastic! Exceeded all expectations",
    "Delivery was late and packaging damaged. Very disappointed.",
    "Best purchase this year. Highly recommend to everyone!",
    "Not worth the price. Feels cheap and poorly made.",
    "It's fine for the price. No complaints but no wow factor either.",
    "Customer service was incredibly helpful and resolved my issue quickly.",
    "Great features but battery life is terrible and overheats frequently."  # Mixed sentiment example
]

# Analyze sentiment
response = client.analyze_sentiment(documents)
results = []
for idx, doc in enumerate(response):
    if not doc.is_error:
        results.append({
            "document": documents[idx],
            "sentiment": doc.sentiment.capitalize(),
            "positive": f"{doc.confidence_scores.positive:.1%}",
            "neutral": f"{doc.confidence_scores.neutral:.1%}",
            "negative": f"{doc.confidence_scores.negative:.1%}",
            "mixed": "Yes" if doc.sentiment == "mixed" else "No"
        })

# Display results in clean tabular format
df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.colheader_justify', 'left')

print("\n" + "="*130)
print("SENTIMENT ANALYSIS RESULTS")
print("="*130)
print(df.to_string(index=False))
print("="*130)
print(f"TOTAL DOCUMENTS ANALYZED: {len(documents)}")

# Calculate summary including Mixed sentiment
sentiment_summary = df['sentiment'].value_counts()
positive = sentiment_summary.get('Positive', 0)
neutral = sentiment_summary.get('Neutral', 0)
negative = sentiment_summary.get('Negative', 0)
mixed = sentiment_summary.get('Mixed', 0)

print(f"SUMMARY: {positive} Positive | {neutral} Neutral | {negative} Negative | {mixed} Mixed")
print("="*130)

# Visualization: Bar chart + Pie chart (including Mixed sentiment)
sentiment_counts = df["sentiment"].value_counts().reindex(
    ["Positive", "Neutral", "Negative", "Mixed"], 
    fill_value=0
)
colors = {
    "Positive": "#27ae60", 
    "Neutral": "#95a5a6", 
    "Negative": "#e74c3c",
    "Mixed": "#f39c12"
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart (left)
valid_sentiments = sentiment_counts[sentiment_counts > 0]
bars = ax1.bar(
    valid_sentiments.index,
    valid_sentiments.values,
    color=[colors.get(s, "#3498db") for s in valid_sentiments.index],
    edgecolor='black',
    linewidth=1.2
)
ax1.set_title("Sentiment Distribution (Bar Chart)", fontsize=13, fontweight="bold", pad=12)
ax1.set_xlabel("Sentiment Category", fontweight="bold", fontsize=11)
ax1.set_ylabel("Document Count", fontweight="bold", fontsize=11)
ax1.grid(axis='y', alpha=0.3, linestyle='--')

for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.15,
             f'{int(height)}',
             ha='center', va='bottom', fontweight='bold', fontsize=11)

# Pie chart (right)
pie_data = valid_sentiments.values
pie_labels = valid_sentiments.index
wedges, texts, autotexts = ax2.pie(
    pie_data,
    labels=pie_labels,
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct*sum(pie_data)/100)})',
    colors=[colors.get(s, "#3498db") for s in pie_labels],
    startangle=90,
    explode=[0.08] * len(pie_data),
    textprops={'fontsize': 10, 'fontweight': 'bold'},
    shadow=True
)
ax2.set_title("Sentiment Proportion (Pie Chart)", fontsize=13, fontweight="bold", pad=12)

plt.tight_layout()
plt.savefig("sentiment_analysis.png", dpi=200, bbox_inches='tight')

# Add description below visualizations
print("\n" + "="*130)
print("VISUALIZATION DESCRIPTION")
print("="*130)
print("• BAR CHART (Left): Shows absolute count of documents per sentiment category.")
print("  - Green: Positive sentiment (enthusiastic, satisfied customers)")
print("  - Gray: Neutral sentiment (factual statements, no strong emotion)")
print("  - Red: Negative sentiment (dissatisfied customers, complaints)")
print("  - Orange: Mixed sentiment (conflicting opinions within same text)")
print("")
print("• PIE CHART (Right): Displays proportional distribution of sentiments across all analyzed documents.")
print("  - Each slice represents percentage share of a sentiment category")
print("  - Values inside slices show both percentage and absolute document count")
print("  - Helps identify dominant sentiment patterns for business decision-making")
print("="*130)
print("\n✓ Visualization saved: sentiment_analysis.png")